# 🍕 Phase 2: Data Science Core with Pizza Store Data

**Complete DS Course** covering:
- M1: Data Lifecycle & CRISP-DM
- M2: Data Cleaning
- M3: Feature Engineering
- M4: Data Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)

In [ ]:
# Create Pizza Store Dataset with messy data for cleaning practice
n_stores = 100

df = pd.DataFrame({
    'store_id': [f'Store_{i+1}' for i in range(n_stores)],
    'city': np.random.choice(['New York', 'Chicago', 'LA', 'Houston', 'Phoenix', None], n_stores, 
                             p=[0.2, 0.2, 0.2, 0.2, 0.15, 0.05]),
    'daily_sales': np.round(np.random.normal(500, 120, n_stores)).astype(float),
    'avg_order_value': np.round(np.random.normal(18, 4, n_stores), 2),
    'customer_rating': np.round(np.random.uniform(2.5, 5.0, n_stores), 1),
    'delivery_time_min': np.round(np.random.exponential(25, n_stores) + 10, 1),
    'num_employees': np.random.randint(5, 25, n_stores),
    'store_size_sqft': np.random.choice([1000, 1500, 2000, 2500], n_stores),
    'has_parking': np.random.choice([True, False], n_stores),
    'years_open': np.random.randint(1, 15, n_stores),
})

# Add missing values
df.loc[np.random.choice(n_stores, 5), 'daily_sales'] = np.nan
df.loc[np.random.choice(n_stores, 3), 'customer_rating'] = np.nan
df.loc[np.random.choice(n_stores, 4), 'delivery_time_min'] = np.nan

# Add duplicates
df = pd.concat([df, df.iloc[[0, 1, 2]]], ignore_index=True)

# Add outliers
df.loc[0, 'daily_sales'] = 2000  # extreme high
df.loc[1, 'daily_sales'] = 10   # extreme low

print(f'Dataset: {len(df)} rows (with issues)')
df.head()

---
## M1: Data Lifecycle & CRISP-DM

**CRISP-DM Phases:**
1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

In [ ]:
# Phase 1: Business Understanding
print("=" * 60)
print("PHASE 1: BUSINESS UNDERSTANDING")
print("=" * 60)
print('''
Business Question: Which factors drive pizza store success?

Goals:
- Identify high-performing stores
- Find factors that predict daily sales
- Recommend improvements for struggling stores

Success Metrics:
- Predict daily sales with R² > 0.5
- Identify top 3 factors affecting sales
''')

In [ ]:
# Phase 2: Data Understanding
print("=" * 60)
print("PHASE 2: DATA UNDERSTANDING")
print("=" * 60)

print("\n📊 Dataset Shape:")
print(f"  Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n📊 Data Types:")
print(df.dtypes)

print("\n📊 Missing Values:")
print(df.isnull().sum())

print("\n📊 Duplicates:")
print(f"  {df.duplicated().sum()} duplicate rows")

print("\n📊 Basic Statistics:")
df.describe()

---
## M2: Data Cleaning

**Common Issues:** Missing values, duplicates, outliers, inconsistent data

### 2.1 Handling Missing Values

In [ ]:
# Check missing values
print("Missing Values Before Cleaning:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

# Strategy 1: Drop rows with missing values
df_dropped = df.dropna()
print(f"\nAfter dropping: {len(df_dropped)} rows (lost {len(df) - len(df_dropped)})")

# Strategy 2: Fill with mean/median/mode
df_clean = df.copy()

# Numeric: fill with median (robust to outliers)
df_clean['daily_sales'].fillna(df_clean['daily_sales'].median(), inplace=True)
df_clean['customer_rating'].fillna(df_clean['customer_rating'].median(), inplace=True)
df_clean['delivery_time_min'].fillna(df_clean['delivery_time_min'].median(), inplace=True)

# Categorical: fill with mode
df_clean['city'].fillna(df_clean['city'].mode()[0], inplace=True)

print("\nMissing Values After Cleaning:")
print(df_clean.isnull().sum())

### 2.2 Removing Duplicates

In [ ]:
# Check duplicates
print(f"Duplicates before: {df_clean.duplicated().sum()}")

# Remove duplicates
df_clean = df_clean.drop_duplicates()
print(f"Duplicates after: {df_clean.duplicated().sum()}")
print(f"Rows remaining: {len(df_clean)}")

### 2.3 Handling Outliers

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower) | (data[column] > upper)]
    return outliers, lower, upper

# Check outliers in daily_sales
outliers, lower, upper = detect_outliers_iqr(df_clean, 'daily_sales')
print(f"Outliers in daily_sales: {len(outliers)}")
print(f"  Lower bound: {lower:.2f}")
print(f"  Upper bound: {upper:.2f}")
print(f"  Outlier values: {outliers['daily_sales'].values}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(df_clean['daily_sales'])
axes[0].set_title('Before Outlier Treatment')
axes[0].set_ylabel('Daily Sales')

# Cap outliers (winsorization)
df_clean['daily_sales'] = df_clean['daily_sales'].clip(lower=lower, upper=upper)

axes[1].boxplot(df_clean['daily_sales'])
axes[1].set_title('After Capping Outliers')
axes[1].set_ylabel('Daily Sales')
plt.tight_layout()
plt.show()

---
## M3: Feature Engineering

**Goal:** Create new features that improve model performance

### 3.1 Creating New Features

In [ ]:
# Feature 1: Sales per employee (efficiency metric)
df_clean['sales_per_employee'] = df_clean['daily_sales'] / df_clean['num_employees']

# Feature 2: Sales per sqft (space efficiency)
df_clean['sales_per_sqft'] = df_clean['daily_sales'] / df_clean['store_size_sqft']

# Feature 3: Is high performer (binary)
median_sales = df_clean['daily_sales'].median()
df_clean['is_high_performer'] = (df_clean['daily_sales'] > median_sales).astype(int)

# Feature 4: Rating category
df_clean['rating_category'] = pd.cut(df_clean['customer_rating'], 
                                      bins=[0, 3, 4, 5], 
                                      labels=['Low', 'Medium', 'High'])

# Feature 5: Store age category
df_clean['store_age_category'] = pd.cut(df_clean['years_open'],
                                         bins=[0, 3, 7, 15],
                                         labels=['New', 'Established', 'Veteran'])

print("New Features Created:")
print(df_clean[['store_id', 'daily_sales', 'sales_per_employee', 'sales_per_sqft', 
                'is_high_performer', 'rating_category', 'store_age_category']].head(10))

### 3.2 Encoding Categorical Variables

In [ ]:
# One-Hot Encoding for city
city_dummies = pd.get_dummies(df_clean['city'], prefix='city')
print("One-Hot Encoding for 'city':")
print(city_dummies.head())

# Label Encoding for rating_category
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_clean['rating_encoded'] = le.fit_transform(df_clean['rating_category'])
print("\nLabel Encoding for 'rating_category':")
print(df_clean[['rating_category', 'rating_encoded']].drop_duplicates())

### 3.3 Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Select numeric features
numeric_features = ['daily_sales', 'avg_order_value', 'delivery_time_min', 'num_employees']

# StandardScaler: mean=0, std=1
scaler_standard = StandardScaler()
df_standard = pd.DataFrame(scaler_standard.fit_transform(df_clean[numeric_features]),
                           columns=[f'{col}_standard' for col in numeric_features])

# MinMaxScaler: range [0, 1]
scaler_minmax = MinMaxScaler()
df_minmax = pd.DataFrame(scaler_minmax.fit_transform(df_clean[numeric_features]),
                         columns=[f'{col}_minmax' for col in numeric_features])

print("Original vs Scaled (first 5 rows):")
comparison = pd.concat([df_clean[numeric_features].head(), 
                        df_standard.head(), 
                        df_minmax.head()], axis=1)
print(comparison.round(2))

---
## M4: Data Visualization

**Goal:** Communicate insights through visual storytelling

### 4.1 Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Histogram
axes[0, 0].hist(df_clean['daily_sales'], bins=20, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Daily Sales')
axes[0, 0].set_xlabel('Daily Sales ($)')

# KDE Plot
df_clean['daily_sales'].plot(kind='kde', ax=axes[0, 1], color='blue')
axes[0, 1].set_title('Density Plot of Daily Sales')
axes[0, 1].set_xlabel('Daily Sales ($)')

# Box Plot by City
df_clean.boxplot(column='daily_sales', by='city', ax=axes[1, 0])
axes[1, 0].set_title('Daily Sales by City')
axes[1, 0].set_xlabel('City')

# Violin Plot
cities = df_clean['city'].dropna().unique()
data_by_city = [df_clean[df_clean['city'] == city]['daily_sales'].values for city in cities]
axes[1, 1].violinplot(data_by_city)
axes[1, 1].set_xticks(range(1, len(cities) + 1))
axes[1, 1].set_xticklabels(cities, rotation=45)
axes[1, 1].set_title('Violin Plot: Sales by City')

plt.tight_layout()
plt.show()

### 4.2 Relationship Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Scatter plot
axes[0].scatter(df_clean['num_employees'], df_clean['daily_sales'], alpha=0.6)
axes[0].set_xlabel('Number of Employees')
axes[0].set_ylabel('Daily Sales ($)')
axes[0].set_title('Employees vs Sales')

# Scatter with hue
for city in df_clean['city'].dropna().unique():
    mask = df_clean['city'] == city
    axes[1].scatter(df_clean.loc[mask, 'num_employees'], 
                    df_clean.loc[mask, 'daily_sales'],
                    label=city, alpha=0.6)
axes[1].set_xlabel('Number of Employees')
axes[1].set_ylabel('Daily Sales ($)')
axes[1].set_title('Employees vs Sales (by City)')
axes[1].legend()

# Rating vs Sales
axes[2].scatter(df_clean['customer_rating'], df_clean['daily_sales'], alpha=0.6, c='green')
axes[2].set_xlabel('Customer Rating')
axes[2].set_ylabel('Daily Sales ($)')
axes[2].set_title('Rating vs Sales')

plt.tight_layout()
plt.show()

### 4.3 Categorical Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Bar plot: Average sales by city
city_sales = df_clean.groupby('city')['daily_sales'].mean().sort_values(ascending=False)
axes[0].bar(city_sales.index, city_sales.values, color='steelblue', edgecolor='black')
axes[0].set_xlabel('City')
axes[0].set_ylabel('Average Daily Sales ($)')
axes[0].set_title('Average Sales by City')
axes[0].tick_params(axis='x', rotation=45)

# Count plot: Stores by rating category
rating_counts = df_clean['rating_category'].value_counts()
axes[1].bar(rating_counts.index, rating_counts.values, color=['red', 'yellow', 'green'], edgecolor='black')
axes[1].set_xlabel('Rating Category')
axes[1].set_ylabel('Number of Stores')
axes[1].set_title('Stores by Rating Category')

# Stacked bar: High performers by city
pivot = df_clean.groupby(['city', 'is_high_performer']).size().unstack(fill_value=0)
pivot.plot(kind='bar', stacked=True, ax=axes[2], color=['salmon', 'lightgreen'])
axes[2].set_xlabel('City')
axes[2].set_ylabel('Number of Stores')
axes[2].set_title('High vs Low Performers by City')
axes[2].legend(['Low Performer', 'High Performer'])
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## Summary

You've learned:
- **Data Lifecycle**: CRISP-DM methodology
- **Data Cleaning**: Missing values, duplicates, outliers
- **Feature Engineering**: Creating, encoding, scaling features
- **Visualization**: Distribution, relationship, and categorical plots